# W51 saturated-pixel recovery: questions and answers

2026-10-21

Questions raised while building the training set, in the order they came up, with the answers we reached. Details, tables and figures are in `report_training_set_audit_20261016.ipynb` (audit) and `report_v2_training_set_20261020.ipynb` (v2 run).

## 1. Does the PSF fitting of saturated stars work, and are its fluxes fit for training?

**Partly, before the fixes.** The fit itself works for isolated stars, but the June training set (480 tables, 2.05 M rows) had two label bugs:

* **Cross-star contamination.** Every saturated pixel inside the 81 px cutout got the fitted star's model flux, including pixels of other stars. Only 40% of rows lay within 5 px of the fitted center; the tail reached 116 px. Fixed by labelling only the fitted source's connected region.
* **`fluxerr` on the wrong scale.** It was stored as a sum of squares (a variance). Fixed by taking the square root. A second error in the same line, found later, is in question 11.

## 2. Can each `sat_level` have the same number of training rows?

**Yes, with a per-source cap.** `sat_level` is the first saturated group (0-4). Level 0 is the scarcest, so it sets the per-level count: about 137,000 rows in the June run. Sampling plain pixels would let a few bright regions dominate (the top 10 sources gave 18% of level 0). Capping rows per (source, level) at 30 and then downsampling each level gave 62,618 rows per level, with the top 10 sources at 0.5% of level 0.

Limits: MIRI has no level-4 rows (question 9), and F560W and F1000W were all level 0 in June. With one common cap, MIRI shrinks to 1-3% of each level, so balance within each instrument.

## 3. What other features could the model use?

Suggested, not implemented:

* PSF model value at the pixel divided by the star's fitted flux (already in memory as `model_image`)
* per-pixel saturation threshold (CRDS `SATURATION`) and non-linearity coefficients (CRDS `LINEARITY`)
* local background level (now stored as `local_bkg`)
* read noise and dark current at the pixel
* amplifier ID, column within the amplifier, detector row and column
* area and compactness of the parent saturated region

## 4. Can we catch failed fits where the saturated region is too large, for example two stars merged into one region?

**Yes, with `area_ratio`.** It is the saturated area divided by the number of model pixels above the cutout's 99th percentile. A clean stellar core gives about 0.2-2. Merged or extended regions in the F1000W calibration exposure gave 9.5 and 14.0. Sources above 8.0 are skipped, and the value is stored as a column.

The 8.0 threshold rests on 12 sources in one exposure. One kept source (6.75) is an elongated streak on a bright ridge, so the cut is not sharp.

## 5. Does a large flux discrepancy in the residual sidelobes, with the check region scaled to the saturated area, flag the failures?

**Not in this field.** Two versions were tested on real sources:

* azimuthal asymmetry of the residual: a merged source scored 0.62, below two clean ones (0.83, 1.07);
* total residual flux in an annulus scaled to the saturated area, in units of noise x sqrt(n): clean sources 10-66, merged ones 13-21, overlapping.

W51's diffuse emission is correlated over many pixels, while the noise x sqrt(n) normalization assumes independent pixels. The statistic is kept as the `sidelobe_residual_snr` column but does not filter. `area_ratio` (question 4) does the filtering.

## 6. Can unsaturated 4QPM pane ramps be the ground truth for saturated imager pixels?

**For part of the pixels.** In full-frame MIRI imaging the coronagraph panes record the same filter at 0.2-0.9 of the imager throughput. Rows with pane truth, by first saturated group (5 MIRI filters): 0 / 4,463 / 62,950 / 99,371 / 101,793 for groups 0-4.

* Group 0: no unsaturated pane anywhere, because the panes saturate there too.
* Groups 2-4: the pipeline slope already agrees with the panes to 2-8%, so pane labels mainly validate.
* What panes add: labels for diffuse emission, where PSF fits fail, and an independent check of PSF labels.
* Limits: +-5-10% pane-imager scatter, bright-end calibration about 6% low, broader PSF, MIRI only.

## 7. Which Detector1 setup should MIRI ramps use?

**The `jwst_new_cal_claude` settings with the `saturation` step on** (`jwst_new_cal_claude_satstep`: `firstframe` and `rscd` skipped, `suppress_one_group=False`). Saturated groups then stay out of the slope fit. NIRCam keeps the original pipeline products, which already ran `saturation`. The v2 training set (`TRAIN_DIR/v2/`, 480 tables, 1.18 M rows, 31,454 sources) uses this setup.

## 8. Why were the v2 MIRI labels first 2-3.5x below the pipeline values?

**The labels lacked the background.** `PSFPhotometry` subtracts a local background (15-30 px annulus) before fitting, so the label held only the star's flux. The ramp records star plus diffuse emission. The label is now PSF model + `local_bkg`, and `local_bkg` is stored. That removed the trend with `sat_level` (MIRI label/`cal` medians went from 0.60 / 0.40 / 0.28 to 0.79 / 0.74 / 0.77 at levels 1 / 2 / 3).

## 9. Why does the training set have no MIRI level-4 rows?

**The `cal` DQ never flags them.** Candidates come from the `cal` DQ `SATURATED` flag. For MIRI that flag matches the ramp exactly for pixels first saturated in groups 0-3, and is never set when only group 4 saturates, because `lastframe` drops group 4 from the fit anyway. Those pixels keep 4 good groups and need no recovery.

## 10. Why did the distance of labelled pixels from the fitted center change between runs?

**Because of the fix in question 1.** June labelled every saturated pixel in the cutout, including other stars' pixels up to 116 px away (median 21.7 px). v2 labels only the fitted star's own connected region (median 2.8 px, 87% within 5 px).

## 11. If the `saturation` step is on, how can a saturated pixel still have a valid `cal` value?

**The step works per group.** When a pixel saturates in group k, groups k and later are flagged in `GROUPDQ`, and the pixel's `cal` DQ also gets `SATURATED`. The pixel **is** flagged. Ramp fitting then fits the slope to groups 0 ... k-1, read before saturation. So `sat_level` = k is also the number of unsaturated groups, and with 2 or more of them the `cal` value is an ordinary measurement.

* Level 0 has no `cal` value; level 1 has a one-group estimate.
* The step also flags each saturating pixel's neighbours (`n_pix_grow_sat=1`), so some candidates are not saturated at all.
* Some `cal` values are NaN because the coronagraph side (x < 360) has no imager flat field (`NO_FLAT_FIELD`), not because of the ramp.

## 12. Where do PSF labels and `cal` values disagree, and does it matter?

**Far from the star's peak, where the label is mostly `local_bkg`.** Tested on rows with `sat_level` 2-4:

* Within 2 px of the fitted center: label/`cal` medians 0.99-1.00 (NIRCam) and 0.94-1.05 (MIRI).
* Where `psf_share` > 0.9 (question 14): 0.99 (NIRCam) and 1.00 (MIRI).
* Far from the peak, the real pixel also holds extended emission, a merged neighbour or a spike, which one PSF plus a flat background cannot model. MIRI runs low (median about 0.6 where `psf_share` < 0.5). NIRCam runs high beyond 10 px (about 2 beyond 25 px).
* NIRCam: 8% of rows outside a factor 1.5, mostly outer pixels, so a small concern. MIRI: about 40% outside a factor 1.5, a real concern. MIRI saturated regions are larger; F1000W and F1280W are worst.

A second `fluxerr` bug turned up here: the stored value is the true error x `psf_star_flux` (fixed in `a7cab72`; divide the v2 `fluxerr` by `psf_star_flux`). Corrected, the per-pixel error is 0.2-0.8% of the label. The observed 10-15% scatter is therefore systematic and `fluxerr` does not capture it.

## 13. What are the next steps, in more detail?

Suggested order:

1. **`fluxerr`**: divide the v2 values by `psf_star_flux` (free), or regenerate.
2. **Label selection**: keep rows with `psf_share` > 0.75 (keeps NIRCam 0.97 / 0.83 / 0.72 / 0.65 / 0.63 and MIRI 0.84 / 0.55 / 0.32 / 0.18 of rows at levels 0-4). A 3 px distance cut keeps fewer rows and treats NIRCam and MIRI PSFs alike.
3. **Level 0**: NIRCam group-0 DN spans 3,800-52,800 and correlates with the label (Spearman 0.78), because a NIRCam group averages several frames, so keep it. MIRI level 0 sits mostly on the plateau (median DN0 58,900); its low-DN0 tail is probably grown neighbours. Keep only below-plateau MIRI rows, or drop them. A correlation is not proof of predictive power; the trained model's level-0 validation error is the test.
4. **Levels 2-4**: decide whether the `cal` value replaces the PSF label (a direct measurement) or stays as a validation column.
5. **Balance and split**: cap rows per (source, level), balance per instrument, and split train and test by source.
6. Later: delete or rewrite the "flux change > 20%" check (it never fires), exclude only HOT pixels instead of whole sources (now 18% of candidates dropped), review `area_ratio` 5-10, add pane-truth rows for MIRI.

## 14. What is `psf_share`?

The fraction of a label that comes from the PSF model rather than the background:

```
label     = flux_fit x PSF(pixel) + local_bkg        (the `flux` column)
psf_share = (label - local_bkg) / label = flux_fit x PSF(pixel) / label
```

`flux_fit` is the star's fitted total flux, `PSF(pixel)` the normalized stpsf model at that pixel, and `local_bkg` the background photutils measured in a 15-30 px annulus. Near 1, the label is almost all fitted star light and is reliable. Near 0, the label is mostly the single background value. It exceeds 1 when `local_bkg` is negative. It is computed from the `flux` and `local_bkg` columns and is not stored in the tables.